# 11.4 同一張圖片，換問題後應怎麼改答案？


同一張紅圓圖，問顏色希望答「紅」，問形狀希望答「圓形」。圖片沒變，問題決定要取哪項資訊；這才是圖片問答，而不只是每次念出完整描述。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-11-same-image-questions.svg")))

In [ ]:
records = [
    {"image": "red_circle", "question": "什麼顏色？", "answer": "紅"},
    {"image": "red_circle", "question": "什麼形狀？", "answer": "圓形"},
]
for row in records:
    print(row["image"], row["question"], "→", row["answer"])
print("同圖", records[0]["image"] == records[1]["image"])
print("答案不同", records[0]["answer"] != records[1]["answer"])

兩筆記錄的 image 身份相同、答案不同，兩項比較都印 True。image 欄的 `red_circle` 是資料名稱，不是已送進模型的像素；實際資料要同時保存圖或可重建它的生成條件，不能只教問答文字便宣稱模型看圖。

訓練時把圖、問題與正確短答保持成對，回答位置算代價。若生成器有隨機選擇，也記種子、版本與參數，才能重建材料。同原圖的不同問題一起分到同一側，避免把看過的圖換問法當新圖。

再固定問題、換圖片：紅圓與藍圓都問顏色，答案應分別紅與藍。同圖多問檢查選擇範圍，同問換圖檢查圖片作用。若只有紅圓多問，模型可能靠「顏色題答紅」猜中。

回答「紅色圓形」雖包含正確詞，對只要顏色的問題仍多答了內容。先訂短答規則，再評內容與範圍；練習加一筆「同時給顏色與形狀」，才把目標改成「紅色圓形」。

<details>
<summary>回顧與查證</summary>

可回顧：[11.3圖片描述目標](https://birdhackor.github.io/tiny-perceptron-vlm/11.3.html)、[7.1對話資料](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vqa.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
